In [1]:
# Installed-package mode. Requires the Atlas14HyetographGrid API.
from pathlib import Path
import ras_commander
print("ras-commander", ras_commander.__version__)

ras-commander 0.122.0


Development mode: install this checkout in your environment from a terminal with
`uv pip install -e ".[notebooks]"`. Restart the kernel afterward. Do not activate a
second import-path override. The shared-pattern API must be present in your installed
version; this notebook accompanies its introduction.

# Atlas 14 Shared Temporal Pattern for an Existing Model

Apply **one NOAA temporal curve scaled by each cell's storm-total depth** to an
existing 2D HEC-RAS model. This produces spatial variation in rainfall depth while
holding the normalized temporal pattern constant. It differs from notebooks
[726](726_abm_hyetograph_grid.ipynb) and
[727](727_atlas14_gridded_rain_on_grid_hecras.ipynb), which construct alternating
blocks from **multiple duration depths**. Neither method is a universal substitute
for the other; choose the method required by the study.

Workflow: select a plan and read its footprint → crop an original NOAA ASCII depth
raster → generate interval-depth NetCDF → check totals, masks and time bounds →
serialize intervals as Float32 → explicitly bilinear-project to SHG DSS, with HEC-Vortex
or with the native FAST-DSS importer → inspect the resulting grids → optionally attach them to a new plan in a
staged project copy → optionally qualify native preprocessing and its materialized
forcing. This notebook never runs the final hydraulic solver.

The native NetCDF is the preferred retained design-storm product. DSS is a receiving
model export. The historical-reproduction detail is **Float32 per interval before
spatial reprojection**; projecting the total first can change the final floating-point
values. Matching another delivery also requires its exact grid extent, CRS, interval
bounds, NoData convention, and numerical runtime. This notebook does not promise
bit-identical results for an arbitrary target model.

## Inputs and dependencies

Use an existing model with a geometry HDF and a known CRS. Read the NOAA raster
metadata to establish scale, frequency basis and CRS; do not infer them from a name.
The example settings below describe Texas 100-year/24-hour PDS data, whose original
ASCII values are thousandths of an inch. Select the temporal region from the NOAA
regional map for **your** model; a state name alone does not select the region.
For a domain crossing temporal regions, one shared curve is an explicit modeling
assumption, not an automatic regional mosaic.

Install `ras-commander[notebooks]` and Rasterio in the same environment.

**Public preset.** With `USE_PUBLIC_PRESET = True` (the setting saved here) the
notebook runs on public data only: the NewOrleansMetro example project from
`RasExamples` and NOAA Atlas 14 Volume 9's 100-year 24-hour PDS grid
(`se100yr24ha.zip`), with the Volume 9 region 1 temporal curve. The preset is a
mechanics demonstration of the workflow, not a regional rainfall study: the region
is not checked against NOAA's regional map for the model, and the small model needs
a 50 % crop buffer for bilinear edges. Set `USE_PUBLIC_PRESET = False` and edit the
configuration cell for your own model.

DSS export takes one of two routes, chosen with `DSS_EXPORT_ROUTE`:

- `"native"` (default) runs the gridded importer of the optional
  `ras-commander-fast-dss` package (FAST-DSS) in-process through
  `RasDssGrid.import_gridded`, with no Java, Jython or GDAL. The notebook states the
  output lattice as an envelope: the model footprint in SHG, snapped outward to whole
  2,000 m cells, plus `FORCING_MARGIN_CELLS`. The public preset uses this route unless
  the environment variable `RAS_COMMANDER_731_EXPORT_ROUTE` says `vortex`. See
  [Native Rust Engines](https://rascommander.info/ras/native-engine/#gridded-import-and-tools).
- `"vortex"` runs HEC-Vortex through `VortexCli`, which needs HEC-Vortex and its
  Jython runtime. Vortex takes GDAL's suggested output extent and then rounds its
  origin to whole cells, which moves the warped values. On the public preset this
  leaves 7 lattice cells missing, so the missing-cell check in section 5 stops the
  notebook there.

**Correction (2026-11-14).** Earlier runs of this notebook exported with Vortex, or
with the native importer's `align="vortex"` option that reproduces it. Both wrote
the same 144 records of 13 × 11 cells, with 7 corner cells missing in every record.
Those cells are not NOAA NoData: the NOAA grid has values there. They are lattice
cells left outside the cropped source after the origin was rounded. The rounding also
moved every value: against a bilinear warp of the complete NOAA grid onto the same
lattice, the stored storm totals differed by up to 0.050 in (RMSE 0.018 in, on totals
near 14 in). The model footprint was covered and no model cell was missing, but the
values were displaced. Treat those earlier DSS files as execution and read-back
evidence that failed source-spatial qualification. The native route now writes to a
stated envelope and is checked against the full-source warp.

Inspection uses `RasDss`, on whichever DSS backend is selected (the HEC-DSS Java
Monolith backend by default). See the precipitation guide for DSS setup and acknowledgments.
The built-in backend does not use pydsstools.

NOAA sources: [depth raster GIS downloads](https://hdsc.nws.noaa.gov/pfds/pfds_gis.html),
[Texas temporal distributions](https://hdsc.nws.noaa.gov/pfds/pfds_temporal.html).
HEC documents [frequency grids with a temporal pattern](https://www.hec.usace.army.mil/confluence/hmsdocs/hmsguides/meteorologic-methods-in-hms/flow-frequency-simulation-options-in-hec-hms/applying-depth-area-analysis-to-precipitation-frequency-grids)
and [alternating-block frequency storms](https://www.hec.usace.army.mil/confluence/hmsdocs/hmstrm/meteorology/precipitation/frequency-storm).

In [2]:
from datetime import datetime, timedelta
from dataclasses import asdict, fields
import importlib.metadata
import json
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from pyproj import CRS
from ras_commander import init_ras_project, RasPlan, RasUnsteady, RasMap, RasPreprocess
from ras_commander.RasProject import stage_project
from ras_commander.hdf import HdfMesh, HdfLandCover, HdfPlan
from ras_commander.precip import Atlas14HyetographGrid, VortexCli
from ras_commander import RasExamples
from ras_commander.dss import RasDss, RasDssGrid
import logging
# hms-commander's Atlas 14 helper logs its cache file's full path at INFO.
logging.getLogger("hms_commander").setLevel(logging.WARNING)

# Edit these paths and choices for your existing model. Inputs stay read-only.
SOURCE_PROJECT = Path("C:/Models/MyModel")
SOURCE_PLAN = "01"
RAS_VERSION = "6.6"  # Receiving runtime; document any upgrade from the source version.
ASC_PATH = Path("C:/Rainfall/tx100yr24ha.asc")
SOURCE_CRS = "EPSG:4269"
SCALE_TO_INCHES = 0.001
DEPTH_BASIS = "PDS"
ARI_YEARS, DURATION_HOURS, INTERVAL_MINUTES = 100, 24, 30
STATE, REGION, QUARTILE, PROBABILITY = "tx", 3, "All Cases", "50%"
# 50% is a temporal-table column, NOT a peak-placement parameter or event AEP.
SOURCE_BUFFER_PERCENT = 10.0  # Crop buffer around the model; a small model may need more for bilinear edges.
SOURCE_URL = "https://hdsc.nws.noaa.gov/pub/hdsc/data/tx/tx100yr24ha.zip"

repo_root = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()
RUN_ROOT = repo_root / "working" / "example_731_shared_temporal"
RUN_ROOT.mkdir(parents=True, exist_ok=True)
RUN_DSS_EXPORT = False
ATTACH_TO_COPIED_MODEL = False
RUN_NATIVE_PRECOMPUTE = False  # Explicit Windows precompute qualification; this never runs the final solver.
PRECOMPUTE_MAX_WAIT = 600
MATERIALIZED_HDF_NODATA_POLICY = None  # Set to 'zero' or 'nan' only after observing the receiving runtime.
REPLACE_LEGACY_PRECIPITATION = False  # Explicitly replace one reviewed area-rainfall boundary on the copy.
SANITIZE_LANDCOVER_NAMES = False  # Optional native 6.6 labels-only compatibility migration on the copy.
PAD_DRY_TAIL = False  # Explicitly opt in if the plan continues after the storm.
DSS_EXPORT_ROUTE = "native"  # "native" (ras-commander-fast-dss, no Java) or "vortex" (HEC-Vortex).
FORCING_MARGIN_CELLS = 1  # Whole 2,000 m cells added around the snapped model footprint.
FORCING_TOLERANCE_IN = 0.005  # Allowed |DSS total - full-source bilinear warp| per lattice cell.
VORTEX_VERSION = ""  # Vortex route: record the installed release before enabling export.
VORTEX_PATH = None    # Or Path("C:/Tools/vortex")

### Public preset

Replaces the configuration above with the public NewOrleansMetro project and the
NOAA Volume 9 grid, both downloaded into the run folder. The original example
archive and the NOAA source are never modified. The receiving model is not
attached or computed in the preset: attaching needs a reviewed choice among the
model's existing rainfall boundaries (see section 6).

In [3]:
USE_PUBLIC_PRESET = True

if USE_PUBLIC_PRESET:
    import os
    import shutil
    import urllib.request
    import zipfile

    RUN_ROOT = repo_root / "working" / "example_731_public_preset"
    if RUN_ROOT.exists():
        shutil.rmtree(RUN_ROOT)  # evidence of a previous preset run is not reused
    RUN_ROOT.mkdir(parents=True)
    SOURCE_PROJECT = RasExamples.extract_project("NewOrleansMetro", output_path=RUN_ROOT / "model")
    SOURCE_PLAN, RAS_VERSION = "01", "7.0"
    SOURCE_URL = "https://hdsc.nws.noaa.gov/pub/hdsc/data/se/se100yr24ha.zip"
    archive = RUN_ROOT / "noaa" / "se100yr24ha.zip"
    archive.parent.mkdir(parents=True)
    with urllib.request.urlopen(SOURCE_URL, timeout=300) as response:
        archive.write_bytes(response.read())
    with zipfile.ZipFile(archive) as opened:
        opened.extractall(archive.parent)
    ASC_PATH = archive.parent / "se100yr24ha.asc"
    print("NOAA grid:", ASC_PATH.name, "|", (archive.parent / "se100yr24ha.prj").read_text().split(",")[0])
    SOURCE_CRS = "EPSG:4269"   # se100yr24ha.prj: GCS_North_American_1983
    SCALE_TO_INCHES = 0.001    # NOAA ASCII values are thousandths of an inch
    STATE, REGION, QUARTILE, PROBABILITY = "se", 1, "All Cases", "50%"
    SOURCE_BUFFER_PERCENT = 50.0
    RUN_DSS_EXPORT = True
    PAD_DRY_TAIL = True
    DSS_EXPORT_ROUTE = os.environ.get("RAS_COMMANDER_731_EXPORT_ROUTE", "native")
    VORTEX_VERSION = os.environ.get("RAS_COMMANDER_731_VORTEX_VERSION", "")
    print("Project:", SOURCE_PROJECT.name, "| plan", SOURCE_PLAN, "| export route:", DSS_EXPORT_ROUTE)

2026-11-14 10:19:23 - ras_commander.RasExamples - INFO - Downloading special project 'NewOrleansMetro'...


2026-11-14 10:19:56 - ras_commander.RasExamples - INFO - Successfully extracted special project 'NewOrleansMetro' to NewOrleansMetro


NOAA grid: se100yr24ha.asc | GEOGCS["GCS_North_American_1983"
Project: NewOrleansMetro | plan 01 | export route: native


## 1. Select the model and its forcing footprint

Use the plan DataFrame to resolve the geometry. Crop in the **ASCII source CRS**;
keep a spatial buffer for bilinear interpolation near the perimeter. The plan's
simulation start anchors this synthetic design event, not an observed storm date.
### Optional public eBFE example

The public East Galveston Bay model can be obtained through `RasExamples`:

```python
from ras_commander import RasExamples
organized = RasExamples.organize_ebfe_model(
    "east-galveston-bay", output_path=Path("C:/Models/eBFE"),
    extract_ras_nested=True, include_results=False,
)
SOURCE_PROJECT = organized / "RAS Model" / "EastGalvBay.prj"
SOURCE_PLAN = "03"
RAS_VERSION = "6.6"
PAD_DRY_TAIL = True
REPLACE_LEGACY_PRECIPITATION = True
SANITIZE_LANDCOVER_NAMES = True
MATERIALIZED_HDF_NODATA_POLICY = "zero"  # Observed on this qualified 6.6 receiving run.
```

The source download is about 10.7 GB and requires a Deflate64-capable 7-Zip
installation. Delivered result archives stay unexpanded by default. Plan 03 is
`1PAC`, a 48-hour HEC-RAS 5.0.7 model with an HMS precipitation-excess boundary.
Using 6.6 is an explicit receiving-runtime upgrade. The replacement below supplies
NOAA gross rainfall, not the original excess rainfall; review infiltration/losses
before using resulting flood levels for engineering decisions. This public eBFE
source is distinct from the RBFS East Galveston deliverable. Download and
organization do not qualify a model for computation.

The delivered land-cover names contain slashes rejected by the 6.6 geometry
preprocessor. The explicit compatibility option uses HEC-RAS's native sanitizer
on the copied sidecar, records its V1-to-V2 serialization migration, and verifies
unchanged class codes, numerical parameters and companion raster bytes. It does
not rebuild or resample the land-cover raster. This optional API is qualified for
the 6.6 runtime only.


In [4]:
ras = init_ras_project(SOURCE_PROJECT, ras_version=RAS_VERSION, load_results_summary=False)
rows = ras.plan_df.loc[ras.plan_df.plan_number.astype(str).str.zfill(2).eq(SOURCE_PLAN)]
assert len(rows) == 1, "Select exactly one existing plan"
plan = rows.iloc[0]
assert pd.notna(plan["unsteady_number"]), "An unsteady-flow plan is required"
geometry_hdf = Path(str(plan["Geom Path"]) + ".hdf")
assert geometry_hdf.is_file(), "Existing geometry HDF required; no preprocessing is run"
footprint = HdfMesh.get_mesh_areas(geometry_hdf)
assert not footprint.empty and footprint.crs is not None
source_bounds = footprint.to_crs(SOURCE_CRS).total_bounds.tolist()
parts = [part.strip() for part in str(plan["Simulation Date"]).split(",")]
assert len(parts) == 4, "Expected start date/time and end date/time"
plan_times = []
for date_text, clock_text in [(parts[0], parts[1]), (parts[2], parts[3])]:
    clock_text = clock_text.replace(":", "")
    assert len(clock_text) == 4 and clock_text.isdigit(), "Expected HHMM plan clock"
    hour, minute = int(clock_text[:2]), int(clock_text[2:])
    assert (0 <= hour < 24 and 0 <= minute < 60) or (hour == 24 and minute == 0)
    plan_times.append(datetime.strptime(date_text, "%d%b%Y") + timedelta(hours=hour, minutes=minute))
simulation_start, simulation_end = plan_times
storm_end = simulation_start + timedelta(hours=DURATION_HOURS)
assert simulation_end >= storm_end, "Plan must span the entire design storm"
print(footprint[["mesh_name"]])
print("Source-CRS bounds:", source_bounds)
print("Storm:", simulation_start, "through", storm_end)

2026-11-14 10:20:00 - ras_commander.RasPrj - INFO - ras-commander v0.122.0 | An open-source project of CLB Engineering Corporation (https://clbengineering.com/) | Docs: https://rascommander.info | GitHub: https://github.com/gpt-cmdr/ras-commander


2026-11-14 10:20:00 - ras_commander.RasPrj - INFO - Project initialized: NewOrleansMetro


2026-11-14 10:20:00 - ras_commander.RasPrj - INFO - Using HEC-RAS version 7.0


2026-11-14 10:20:00 - ras_commander.RasPrj - INFO - 
═══════════════════════════════════════════════════════════════════════
ras-commander | HEC-RAS Automation Library
Docs: https://rascommander.info/ras/
Repo: https://github.com/gpt-cmdr/ras-commander
LLM agents: https://rascommander.info/ras/llms.txt
═══════════════════════════════════════════════════════════════════════

SUPPORT OPEN-SOURCE DEVELOPMENT:
  Using RAS Commander? Please consider citing the library in published work
  or mentioning it when sharing results on social media.
  Agents are encouraged to suggest a RAS Commander v0.122.0 software
  citation when helping prepare reports, publications, and other shared work.
  Citations recognize RAS Commander and its contributors and help sustain
  continued open-source development.
  Citation and sharing guidance: https://rascommander.info/ras/cite/

PROJECT DATAFRAMES (single source of truth — use these, not file globbing):
  ras.plan_df        Plans, HDF paths, geometry/flow 

          mesh_name
0  NewOrleans Metro
Source-CRS bounds: [-90.1686239827324, 29.91505706283603, -90.06119062897804, 30.027319932154434]
Storm: 2024-10-02 00:00:00 through 2024-10-03 00:00:00


## 2. Generate the shared-pattern design storm

For native cell depth $D(x,y)$ and cumulative NOAA fractions $F(t)$, each interval is
$P_i(x,y)=D(x,y)[F(t_{i+1})-F(t_i)]$. No alternating-block rearrangement, areal
reduction, or PDS/AMS conversion is applied. Intensity is interval depth divided by
interval hours. A 100-year rainfall frequency does not establish a 1% flood AEP.

The ASCII route keeps source identity, CRS, scale and NoData explicit. The alternative
`Atlas14HyetographGrid.generate(bounds=..., ...)` uses the remote grid service;
its WGS84 bounds and source product must not be substituted silently when reproducing
an original ASCII workflow.

In [5]:
raw_nc = Atlas14HyetographGrid.generate_from_asc_file(
    ASC_PATH, scale_factor=SCALE_TO_INCHES, source_crs=SOURCE_CRS,
    ari_years=ARI_YEARS, storm_duration_hours=DURATION_HOURS,
    timestep_minutes=INTERVAL_MINUTES, state=STATE, region=REGION,
    quartile=QUARTILE, probability_column=PROBABILITY,
    bounds=source_bounds, buffer_percent=SOURCE_BUFFER_PERCENT, depth_frequency_basis=DEPTH_BASIS,
    output_netcdf=RUN_ROOT / "shared_pattern_native.nc",
)
with xr.open_dataset(raw_nc, engine="scipy", decode_times=False) as opened:
    storm = opened.load()
print(storm)

2026-11-14 10:20:01 - ras_commander.precip.Atlas14HyetographGrid - INFO - Wrote shared-pattern Atlas 14 grid shared_pattern_native.nc (48 intervals, 702 valid cells)


<xarray.Dataset> Size: 547kB
Dimensions:             (time: 48, y: 27, x: 26, bounds: 2)
Coordinates:
  * time                (time) float64 384B 0.5 1.0 1.5 2.0 ... 23.0 23.5 24.0
  * y                   (y) float64 216B 30.08 30.07 30.07 ... 29.88 29.87 29.87
  * x                   (x) float64 208B -90.22 -90.21 -90.2 ... -90.02 -90.01
  * bounds              (bounds) int8 2B 0 1
Data variables:
    precip_incremental  (time, y, x) float64 270kB 0.1817 0.182 ... 0.001439
    precip_cumulative   (time, y, x) float64 270kB 0.1817 0.182 ... 14.39 14.39
    time_bounds         (time, bounds) float64 768B 0.0 0.5 0.5 ... 23.5 24.0
    temporal_fraction   (time) float64 384B 0.0132 0.0169 ... 0.0016 0.0001
    storm_total         (y, x) float64 6kB 13.77 13.79 13.82 ... 14.39 14.39
    spatial_ref         int32 4B 0
Attributes: (12/28)
    title:                                Atlas 14 shared-temporal-pattern pr...
    Conventions:                          CF-1.8
    method_name:         

## 3. Check conservation, missing cells and synchronization

Missing cells remain missing. Do not use a NaN-skipping sum that turns an entirely
missing cell into a dry cell. Native-grid storm totals and normalized curves should
agree before any reprojection. Reprojection can change pointwise totals at new cell
locations; it must be evaluated on its own grid.

In [6]:
increments = storm.precip_incremental.values
native_total = storm.storm_total.values
fractions = storm.temporal_fraction.values
bounds_hours = storm.time_bounds.values
valid = np.isfinite(native_total)
assert valid.any()
np.testing.assert_array_equal(np.isfinite(increments), np.broadcast_to(valid, increments.shape))
assert (increments[:, valid] >= 0).all()
np.testing.assert_allclose(increments.sum(axis=0), native_total, rtol=1e-12, atol=1e-12, equal_nan=True)
np.testing.assert_allclose(increments[:, valid], fractions[:, None] * native_total[valid], rtol=1e-12, atol=1e-12)
np.testing.assert_allclose(fractions.sum(), 1.0, rtol=0, atol=1e-12)
np.testing.assert_allclose(np.diff(bounds_hours, axis=1), INTERVAL_MINUTES / 60)
np.testing.assert_allclose(bounds_hours[1:, 0], bounds_hours[:-1, 1])
np.testing.assert_allclose(bounds_hours[[0, -1], [0, 1]], [0, DURATION_HOURS])
np.testing.assert_allclose(storm.time.values, bounds_hours[:, 1])
assert CRS.from_wkt(storm.spatial_ref.attrs["crs_wkt"]).equals(CRS.from_user_input(SOURCE_CRS))
print("Native QC passed:", increments.shape[0], "intervals;", valid.sum(), "valid cells")

Native QC passed: 48 intervals; 702 valid cells


In [7]:
# Plot actual cell coordinates and interval boundaries, not array-index timestamps.
row, col = np.argwhere(valid)[len(np.argwhere(valid)) // 2]
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
mesh = axes[0].pcolormesh(storm.x, storm.y, native_total, shading="auto", cmap="Blues")
footprint.to_crs(SOURCE_CRS).boundary.plot(ax=axes[0], color="black", linewidth=1)
fig.colorbar(mesh, ax=axes[0], label="Storm total (in)")
axes[0].set_title("NOAA cell depths and model footprint")
axes[0].set_xlabel("Source CRS x"); axes[0].set_ylabel("Source CRS y")
dt = INTERVAL_MINUTES / 60
axes[1].bar(bounds_hours[:, 0], increments[:, row, col] / dt, width=dt, align="edge")
axes[1].set(xlabel="Hours since storm start", ylabel="Intensity (in/hr)", title="Sample-cell hyetograph")
axes[2].plot(np.r_[0, bounds_hours[:, 1]], np.r_[0, np.cumsum(increments[:, row, col])])
axes[2].set(xlabel="Hours since storm start", ylabel="Cumulative depth (in)", title="Same sample cell and clock")
plt.tight_layout()

## 4. Prepare Float32 interval NetCDF with the model clock

Serialize each native-grid **interval depth**, then let the importer (HEC-Vortex or
FAST-DSS) reproject each frame.
Do not reproject the storm-total grid first. Keep the numeric time bounds and their
units together. No leading missing interval is invented: normal design rainfall
starts at the model start. Reproducing a delivery with a leading missing record or
offset clock is a separate compatibility choice that must be documented explicitly.

If the source plan outlasts the storm, `PAD_DRY_TAIL=True` appends explicit dry
intervals at valid cells while retaining spatial NoData. This assumes no additional
rain after the design event. Otherwise export stops rather than shortening the plan.

**Vortex classification:** set the interval variable's `long_name="precipitation"`
and `standard_name="lwe_thickness_of_precipitation_amount"` explicitly. Vortex
0.13.3 classifies the variable before applying the requested DSS C-part; setting
C=`PRECIPITATION` alone can still produce a `PER-AVER` fallback for an unrecognized
variable. Retain CF `cell_methods="time: sum"` and verify `IN`/`PER-CUM` after export.
See [Vortex DSS writer](https://github.com/HydrologicEngineeringCenter/Vortex/blob/v0.13.3/vortex-api/src/main/java/mil/army/usace/hec/vortex/io/DssDataWriter.java).


In [8]:
forcing = storm[["precip_incremental", "time_bounds", "spatial_ref"]].copy(deep=True)
end_hours = (simulation_end - simulation_start).total_seconds() / 3600
if end_hours > DURATION_HOURS:
    if not PAD_DRY_TAIL:
        raise ValueError("Plan continues after storm: explicitly enable PAD_DRY_TAIL or review the plan window")
    count = int(round(end_hours / dt))
    assert np.isclose(count * dt, end_hours, rtol=0, atol=1e-9), "Plan end must align with the rainfall interval"
    all_bounds = np.column_stack((np.arange(count) * dt, (np.arange(count) + 1) * dt))
    padded = np.full((count, *native_total.shape), np.nan, dtype=np.float64)
    padded[:, valid] = 0.0
    padded[:len(increments)] = increments
    forcing = forcing.drop_dims("time").assign_coords(time=all_bounds[:, 1])
    forcing["precip_incremental"] = (("time", "y", "x"), padded, dict(storm.precip_incremental.attrs))
    forcing["time_bounds"] = (("time", "bounds"), all_bounds)
forcing["precip_incremental"] = forcing.precip_incremental.astype("float32")
forcing.precip_incremental.attrs.update(units="in", cell_methods="time: sum", long_name="precipitation", standard_name="lwe_thickness_of_precipitation_amount", grid_mapping="spatial_ref")
clock_units = "hours since " + simulation_start.strftime("%Y-%m-%d %H:%M:%S")
forcing.time.attrs.update(units=clock_units, calendar="standard", bounds="time_bounds", axis="T", standard_name="time")
forcing.time_bounds.attrs.update(units=clock_units, calendar="standard")
forcing.attrs.update(source_url=SOURCE_URL, spatial_resampling="Bilinear", target_grid="SHG, 2000 m", precision_order="native interval -> float32 -> spatial reprojection")
float32_nc = RUN_ROOT / "shared_pattern_float32_model_clock.nc"
assert not float32_nc.exists(), "Use a new run folder rather than overwriting evidence"
forcing.to_netcdf(float32_nc, engine="scipy")
with xr.open_dataset(float32_nc, engine="scipy", decode_times=False) as check:
    np.testing.assert_array_equal(check.precip_incremental.values, forcing.precip_incremental.values)
    np.testing.assert_array_equal(check.time_bounds.values, forcing.time_bounds.values)
    assert check.precip_incremental.dtype == np.dtype("float32")
provenance = dict(storm.attrs)
provenance.update(source_url=SOURCE_URL, plan=SOURCE_PLAN, source_project=str(SOURCE_PROJECT),
                  simulation_start=str(simulation_start), simulation_end=str(simulation_end),
                  resampling="Bilinear", target_cell_size_m=2000, float32_before_warp=True,
                  areal_reduction="none", dry_tail=PAD_DRY_TAIL)
(RUN_ROOT / "provenance.json").write_text(json.dumps(provenance, indent=2, default=str), encoding="utf-8")

2131

## 5. Optional DSS export and read-back

Set `RUN_DSS_EXPORT=True` only after the native QC is satisfactory. Bilinear is
explicit: underlying programmatic Vortex callers can otherwise select a different
fallback. Vortex determines the output extent from the input domain here; an exact
delivery comparison additionally needs the delivered grid origin and extent.
Record the installed Vortex version with the run evidence.

With `DSS_EXPORT_ROUTE = "native"`, the same NetCDF goes through
`RasDssGrid.import_gridded` (the RAS Commander wrapper of the FAST-DSS importer)
with the same target (SHG, 2,000 m, bilinear) and pathname parts, onto a stated
envelope: the model footprint snapped outward to whole cells plus
`FORCING_MARGIN_CELLS`. It writes the records in this process and is recorded in the
provenance with the installed package version. It refuses to move a grid the way
Vortex does; see [intended differences](https://rascommander.info/ras/native-engine/#dss-intended-differences).
The read-back checks below are the same for both routes.

**Missing cells and spatial checks.** Every lattice cell is counted, and any missing
cell stops the export: a NaN-skipping sum would read it as a dry cell. The stored storm
total is compared with a GDAL bilinear warp of the **complete** NOAA grid (not the
crop) onto the stored lattice, with an exact transformer. The tolerance
`FORCING_TOLERANCE_IN` (0.005 in) allows for GDAL's downsampling kernel, which widens
with the source-to-target scale and so differs slightly between a cropped and a
complete source; a moved grid or a crop that is too small for the lattice exceeds it.
Finally, every physical mesh cell centre, and the four lattice cells around it that a
bilinear interpolation reads, must lie on the lattice on present cells.

Read every resulting record with `RasDss.read_grid`. Check chronology, units, mask,
geometry and totals before attaching it. These are rainfall-data checks, not a
hydraulic validation. The export uses interval **amounts** (`IN`, `PER-CUM`), not
intensity or cumulative-since-storm-start values.

In [9]:
dss_path = RUN_ROOT / "shared_pattern_shg.dss"
dss_parts = {"A": "SHG", "B": "DESIGN", "C": "PRECIPITATION", "F": "ATLAS14_SHARED"}
if RUN_DSS_EXPORT:
    assert DSS_EXPORT_ROUTE in {"vortex", "native"}, "DSS_EXPORT_ROUTE is 'vortex' or 'native'"
    assert not dss_path.exists(), "Use a new DSS file"
    provenance["dss_export_route"] = DSS_EXPORT_ROUTE
    # Stated lattice: model footprint in SHG (CONUS Albers, metres) snapped outward to whole
    # cells, plus a margin so every mesh cell's bilinear stencil lies on the lattice.
    shg_west, shg_south, shg_east, shg_north = footprint.to_crs("EPSG:5070").total_bounds
    margin_m = FORCING_MARGIN_CELLS * 2000
    envelope = (float(np.floor(shg_west / 2000) * 2000 - margin_m), float(np.floor(shg_south / 2000) * 2000 - margin_m),
                float(np.ceil(shg_east / 2000) * 2000 + margin_m), float(np.ceil(shg_north / 2000) * 2000 + margin_m))
    if DSS_EXPORT_ROUTE == "vortex":
        assert VORTEX_VERSION.strip(), "Record the installed Vortex release in VORTEX_VERSION"
        provenance["vortex_version"] = VORTEX_VERSION
        provenance["vortex_path"] = str(VortexCli.find_vortex(VORTEX_PATH))
        (RUN_ROOT / "provenance.json").write_text(json.dumps(provenance, indent=2, default=str), encoding="utf-8")
        VortexCli.import_gridded(
            input_files=[float32_nc], output_dss=dss_path, variables=["precip_incremental"],
            target_wkt="SHG", target_cell_size=2000, resampling_method="Bilinear",
            dss_parts=dss_parts, vortex_path=VORTEX_PATH,
        )
    else:
        # FAST-DSS gridded importer through RasDssGrid: same NetCDF, target, method and parts.
        provenance["fast_dss_version"] = importlib.metadata.version("ras-commander-fast-dss")
        provenance["envelope_shg_m"] = envelope
        (RUN_ROOT / "provenance.json").write_text(json.dumps(provenance, indent=2, default=str), encoding="utf-8")
        # State the lattice. Without an envelope, Vortex (and align="vortex") round GDAL's
        # suggested origin to whole cells, which moves every warped value.
        imported = RasDssGrid.import_gridded(
            [float32_nc], dss_path, variable="precip_incremental",
            target_crs="SHG", cell_size=2000, method="bilinear", parts=dss_parts,
            envelope=envelope,
        )
        print("FAST-DSS wrote", len(imported.records), "grid records on envelope", envelope)
    catalog = RasDss.get_catalog(dss_path)
    frames = [RasDss.read_grid(dss_path, p) for p in catalog.pathname]
    assert frames, "The export produced no grid records"
    ordered = sorted(zip(catalog.pathname, frames), key=lambda pair: pair[1]["start_time"])
    dss_paths, frames = zip(*ordered)
    assert len(frames) == len(forcing.time)
    for i, frame in enumerate(frames):
        expected = [simulation_start + timedelta(hours=float(h)) for h in forcing.time_bounds.values[i]]
        assert pd.Timestamp(frame["start_time"]) == expected[0]
        assert pd.Timestamp(frame["end_time"]) == expected[1]
    origin_keys = ("lower_left_cell", "origin", "projection")
    for frame in frames:
        assert all(frame["metadata"][key] == frames[0]["metadata"][key] for key in origin_keys), "Grid origin changed between intervals"
        assert frame["units"].upper() == "IN" and frame["data_type"].upper() == "PER-CUM"
        assert frame["cell_size"] == 2000 and frame["shape"] == frames[0]["shape"]
        assert CRS.from_user_input(frame["crs"]).equals(CRS.from_user_input(frames[0]["crs"]))
    dss_values = np.stack([frame["data"] for frame in frames]).astype(np.float64)
    # Count missing cells explicitly; never let a NaN-skipping reduction read them as dry.
    missing = ~np.isfinite(dss_values)
    missing_per_record = np.count_nonzero(missing, axis=(1, 2))
    print(f"{len(frames)} records of {frames[0]['shape'][0]} x {frames[0]['shape'][1]} cells; "
          f"missing cells per record: {sorted(set(missing_per_record.tolist()))}; "
          f"missing in every record: {int(missing.all(axis=0).sum())}; in some record: {int(missing.any(axis=0).sum())}")
    if missing.any():
        raise RuntimeError("Missing DSS cells on the forcing lattice; check the crop buffer and the export envelope "
                           "(HEC-Vortex chooses and rounds its own extent)")
    assert dss_values.min() >= 0
    # Public grid arrays are south-first; raster masks are north-first.
    import rasterio
    from rasterio.features import geometry_mask
    from rasterio.transform import from_origin
    from rasterio.warp import reproject, Resampling
    meta = frames[0]["metadata"]
    cell = float(frames[0]["cell_size"])
    left, bottom = meta["origin"]
    height, width = frames[0]["shape"]
    right, top = left + width * cell, bottom + height * cell
    if DSS_EXPORT_ROUTE == "native":
        assert (left, bottom, right, top) == envelope, "Stored lattice differs from the stated envelope"
    projected_footprint = footprint.to_crs(frames[0]["crs"])
    west, south, east, north = projected_footprint.total_bounds
    assert left <= west and bottom <= south and right >= east and top >= north, "DSS must cover the entire model footprint"
    model_cells = geometry_mask(projected_footprint.geometry, out_shape=(height, width),
                                transform=from_origin(left, top, cell, cell), invert=True, all_touched=True)[::-1]
    assert model_cells.any() and np.isfinite(dss_values[:, model_cells]).all(), "No missing rainfall over model cells"
    # Float32 serialization/reprojection allows small rounding differences.
    # Check the shared curve independently of the new spatial cell depths.
    wet = dss_values[:len(increments), model_cells].astype(np.float64)
    projected_total = wet.sum(axis=0)
    np.testing.assert_allclose(wet, fractions[:, None] * projected_total[None, :],
                               rtol=5e-6, atol=1e-7, err_msg="Export changed the shared temporal curve")
    if PAD_DRY_TAIL and len(frames) > len(increments):
        assert np.abs(dss_values[len(increments):]).max() == 0

    # Full-source spatial check: bilinear warp of the complete NOAA grid onto the stored lattice.
    with rasterio.open(ASC_PATH) as source:
        source_depth = source.read(1).astype(np.float64)
        if source.nodata is not None:
            source_depth[source_depth == source.nodata] = np.nan
        source_transform = source.transform
    source_depth *= SCALE_TO_INCHES
    reference = np.full((height, width), np.nan)
    reproject(source_depth, reference, src_transform=source_transform, src_crs=SOURCE_CRS, src_nodata=np.nan,
              dst_transform=from_origin(left, top, cell, cell), dst_crs=frames[0]["crs"], dst_nodata=np.nan,
              resampling=Resampling.bilinear, tolerance=0)
    reference = reference[::-1]  # south-first, like the DSS arrays
    stored_total = dss_values[:len(increments)].sum(axis=0)
    warp_error = stored_total - reference
    print(f"Full-source warp: {int(np.count_nonzero(~np.isfinite(reference)))} reference cells outside NOAA coverage; "
          f"|DSS total - warp| max {np.abs(warp_error).max():.2e} in, RMSE {np.sqrt(np.mean(warp_error ** 2)):.2e} in "
          f"(tolerance {FORCING_TOLERANCE_IN} in)")
    if not np.isfinite(reference).all() or not np.abs(warp_error).max() <= FORCING_TOLERANCE_IN:
        raise RuntimeError("DSS grids do not match the full-source warp; the lattice is outside NOAA coverage, "
                           "the crop buffer is too small, or the route moved the grid (HEC-Vortex rounds its origin)")

    # Every physical mesh cell centre, and its four-centre bilinear stencil, must sit on present cells.
    present = ~missing.any(axis=0)
    mesh_points = HdfMesh.get_mesh_cell_points(geometry_hdf, include_ghost_cells=False).to_crs(frames[0]["crs"])
    point_x, point_y = mesh_points.geometry.x.to_numpy(), mesh_points.geometry.y.to_numpy()
    def on_present(rows, cols):
        inside = (rows >= 0) & (rows < height) & (cols >= 0) & (cols < width)
        return inside & present[rows.clip(0, height - 1), cols.clip(0, width - 1)]
    centre_ok = on_present(np.floor((point_y - bottom) / cell).astype(int), np.floor((point_x - left) / cell).astype(int))
    stencil_ok = np.ones(len(mesh_points), dtype=bool)
    for d_row in (0, 1):
        for d_col in (0, 1):
            stencil_ok &= on_present(np.floor((point_y - bottom) / cell - 0.5).astype(int) + d_row,
                                     np.floor((point_x - left) / cell - 0.5).astype(int) + d_col)
    print(f"Mesh cell centres: {len(mesh_points)}; off the lattice or on a missing cell: {int((~centre_ok).sum())}; "
          f"bilinear stencil touching a missing or off-lattice cell: {int((~stencil_ok).sum())}")
    if not (centre_ok.all() and stencil_ok.all()):
        raise RuntimeError("Some mesh cells would sample missing or off-lattice forcing")

    print("DSS intervals, missing-cell, full-source and mesh checks passed; inspect metadata and spatial coverage:")
    print({k: v for k, v in frames[0].items() if k not in ("data", "dss_file", "path")})
    print("Projected cell totals (in):", stored_total.min(), stored_total.max())
else:
    print("Retained NetCDF ready; DSS export not requested")

FAST-DSS wrote 144 grid records on envelope (558000.0, 774000.0, 576000.0, 794000.0)


144 records of 10 x 9 cells; missing cells per record: [0]; missing in every record: 0; in some record: 0


Full-source warp: 0 reference cells outside NOAA coverage; |DSS total - warp| max 1.66e-03 in, RMSE 1.83e-04 in (tolerance 0.005 in)
Mesh cell centres: 19711; off the lattice or on a missing cell: 0; bilinear stencil touching a missing or off-lattice cell: 0
DSS intervals, missing-cell, full-source and mesh checks passed; inspect metadata and spatial coverage:
{'pathname': '/SHG/DESIGN/PRECIPITATION/02OCT2024:0000/02OCT2024:0030/ATLAS14_SHARED/', 'shape': (10, 9), 'units': 'IN', 'data_type': 'PER-CUM', 'grid_type': 'albers', 'crs': 'PROJCS["USA_Contiguous_Albers_Equal_Area_Conic_USGS_version",GEOGCS["GCS_North_American_1983",DATUM["D_North_American_1983",SPHEROID["GRS_1980",6378137.0,298.257222101]],PRIMEM["Greenwich",0.0],UNIT["Degree",0.0174532925199433]],PROJECTION["Albers"],PARAMETER["False_Easting",0.0],PARAMETER["False_Northing",0.0],PARAMETER["Central_Meridian",-96.0],PARAMETER["Standard_Parallel_1",29.5],PARAMETER["Standard_Parallel_2",45.5],PARAMETER["Latitude_Of_Origin",23.0]

## 6. Optional attachment to a copied model

This step stages a separate project, clones both unsteady flow and plan, and uses
`configure_gridded_dss_precipitation`. It does not edit the original project or
change its simulation window. The staging API checks referenced assets; resolve
missing or external dependencies before treating the copy as portable. Review
inherited boundary flows, infiltration and other meteorology for consistency with
the design event. Conflicting precipitation boundaries must be reviewed explicitly. With
`REPLACE_LEGACY_PRECIPITATION=True`, this example removes exactly one superseded
area-rainfall block from the staged copy before cloning the new flow and plan.
The default refuses removal. Multiple legacy rainfall blocks require a separately
reviewed preparation step; they are not silently deleted. The source tree is
unchanged, while its original flow copy in the staged tree is also superseded.

After attachment, inspect rainfall coverage and totals in the receiving model.
The optional precompute qualification below checks the receiving temporary HDF;
follow the broader qualification checks in notebooks 727/728 before computing. A successful
DSS write is not proof that the hydraulic model consumes it correctly. In particular, inspect interval alignment in
the preprocessed temporary HDF when using HEC-RAS 6.6; period-average grid ingestion
can introduce a one-interval timing shift. Do not compensate by silently shifting
the source timestamps or accept a shifted rainfall series as a passing check.

In [10]:
if ATTACH_TO_COPIED_MODEL:
    assert RUN_DSS_EXPORT and dss_path.is_file(), "Export and inspect the DSS first"
    staged = stage_project(SOURCE_PROJECT, RUN_ROOT / "model_copy")
    removal = None
    # Inspect before any plan/flow edits: deletion uses the staging evidence.
    boundaries = RasUnsteady.inspect_boundary_blocks(staged, unsteady_number=plan["unsteady_number"])
    legacy = boundaries.loc[boundaries["bc_type"].eq("Precipitation Hydrograph")]
    if not legacy.empty:
        print(legacy[["boundary_id", "area_2d", "bc_type", "boundary_location_raw"]].to_string(index=False))
        if not REPLACE_LEGACY_PRECIPITATION or len(legacy) != 1:
            raise ValueError("Review legacy rainfall and explicitly opt in to replacing one block; the source is unchanged.")
        boundary = legacy.iloc[0]
        removal = RasUnsteady.delete_boundary(
            staged, unsteady_number=plan["unsteady_number"],
            boundary_id=boundary["boundary_id"],
            expected_source_sha256=boundary["owner_sha256"],
            expected_block_sha256=boundary["block_sha256"],
            expected_bc_type=boundary["bc_type"],
            expected_location_raw=boundary["boundary_location_raw"], dry_run=False,
        )
        assert removal.state == "applied"
        print("Superseded legacy rainfall on copy:", removal.boundary_location_raw)
    # Staging initially resolves the delivered version. Use the chosen receiver.
    copied = init_ras_project(staged.ras_object.project_folder, RAS_VERSION,
                              ras_object=True, load_results_summary=False)
    landcover_migrations = []
    if SANITIZE_LANDCOVER_NAMES:
        layers = RasMap.list_landcover_layers(copied.project_folder)
        if layers.empty:
            raise ValueError("No registered land-cover layer to sanitize; review the compatibility option.")
        for layer_path in layers["resolved_path"]:
            registered = Path(layer_path)
            if registered.suffix.lower() == ".hdf":
                sidecar, exact_raster = registered, None  # Native HDF association resolves the TIFF.
            elif registered.suffix.lower() in {".tif", ".tiff"}:
                sidecar, exact_raster = registered.with_suffix(".hdf"), registered
            else:
                raise ValueError(f"Unsupported registered land-cover path: {registered}")
            if not sidecar.is_file():
                raise FileNotFoundError(f"Review the native land-cover sidecar: {sidecar}")
            migrated = HdfLandCover.sanitize_classification_names(
                sidecar, raster_path=exact_raster, hecras_version=RAS_VERSION,
            )
            landcover_migrations.append(asdict(migrated))
    new_unsteady = RasPlan.clone_unsteady(plan["unsteady_number"], new_title="Atlas14 shared pattern", ras_object=copied)
    new_plan = RasPlan.clone_plan(SOURCE_PLAN, new_plan_shortid="Atlas14 shared", new_title="Atlas14 shared pattern",
                                  unsteady_flow=new_unsteady, ras_object=copied)
    RasUnsteady.configure_gridded_dss_precipitation(
        new_unsteady, dss_filename=str(dss_path.resolve()), dss_pathname=dss_paths[0],
        ratio=1.0, interpolation="Bilinear", ras_object=copied,
    )
    config = RasUnsteady.get_met_precipitation_config(new_unsteady, ras_object=copied)
    assert config["mode"] == "Gridded" and config["source"] == "DSS"
    configured_dss = Path(config["dss_filename"])
    if not configured_dss.is_absolute():
        configured_dss = copied.project_folder / configured_dss
    assert configured_dss.resolve() == dss_path.resolve()
    assert config["dss_pathname"] == dss_paths[0]
    attachment_receipt = {
        "source_project": str(SOURCE_PROJECT), "source_plan": SOURCE_PLAN,
        "receiving_version": RAS_VERSION, "copied_plan": new_plan, "copied_flow": new_unsteady,
        "staging": {field.name: getattr(staged, field.name) for field in fields(staged)
                    if field.name not in {"assets", "ras_object"}},
        "superseded_boundary": asdict(removal) if removal is not None else None,
        "precipitation_config": config,
        "landcover_name_migrations": landcover_migrations,
    }
    (RUN_ROOT / "attachment-receipt.json").write_text(
        json.dumps(attachment_receipt, indent=2, default=str), encoding="utf-8",
    )
    print("Prepared copied plan:", new_plan, "in", copied.project_folder)
    print("Retain the sibling DSS with this workspace; its reference is absolute.")
else:
    print("Original model unchanged. Attachment is an explicit optional step.")

Original model unchanged. Attachment is an explicit optional step.


## 7. Optional native precompute qualification

Set `RUN_NATIVE_PRECOMPUTE=True` only after attachment and DSS read-back pass. The
public `RasPreprocess.preprocess_plan()` API starts Windows HEC-RAS, waits for its
preprocess handoff, stops the owned process, and validates the completed temporary
HDF. This section opens that HDF **only after** the API returns; never inspect an
actively written HDF from another process.

The read-back binds the materialized interval depths and interval-end timestamps to
the already-reviewed DSS frames. It also checks the receiving configuration, grid
attributes, spatial row order, and an explicitly observed NoData policy. It creates
a precompute receipt for a qualified solver input, not a solver result or an
engineering acceptance decision.


In [11]:
if RUN_NATIVE_PRECOMPUTE:
    assert ATTACH_TO_COPIED_MODEL and RUN_DSS_EXPORT, (
        "Native precompute requires the staged copy and reviewed DSS frames."
    )
    assert MATERIALIZED_HDF_NODATA_POLICY in {"zero", "nan"}, (
        "Set MATERIALIZED_HDF_NODATA_POLICY from an observed receiving-HDF result."
    )
    prepared = RasPreprocess.preprocess_plan(
        new_plan, ras_object=copied, max_wait=PRECOMPUTE_MAX_WAIT,
    )
    assert prepared, prepared.error
    # RasPreprocess has completed its owned Windows lifecycle before this HDF read.
    materialized = HdfPlan.get_plan_met_precip_values(prepared.tmp_hdf_path)
    hdf_values = np.asarray(materialized["values"], dtype=np.float64)
    raw_hdf_times = np.asarray(materialized["timestamps"])
    hdf_times = pd.DatetimeIndex([
        pd.Timestamp(datetime.strptime(
            bytes(value).decode("ascii").strip() if isinstance(value, (bytes, np.bytes_)) else str(value).strip(),
            "%d%b%Y %H:%M:%S.%f",
        ))
        for value in raw_hdf_times
    ])
    expected_ends = pd.DatetimeIndex([pd.Timestamp(frame["end_time"]) for frame in frames])
    assert hdf_values.shape[0] == len(hdf_times) == len(frames)
    assert hdf_values.shape[1] == int(np.prod(frames[0]["shape"]))
    assert hdf_times.equals(expected_ends), "Materialized interval-end timestamps differ from DSS"

    received = materialized["attributes"]
    assert received["Mode"] == "Gridded" and received["Source"] == "DSS"
    assert received["Data Type"].casefold() == "per-cum"
    assert received["Units"].casefold() in {"in", "inch", "inches"}
    assert float(received["Ratio"]) == 1.0
    assert received["Interpolation Method"].casefold() == "bilinear"
    assert received["DSS Pathname"] == config["dss_pathname"] == dss_paths[0]
    received_dss = Path(received["DSS Filename"])
    if not received_dss.is_absolute():
        received_dss = prepared.tmp_hdf_path.parent / received_dss
    assert received_dss.resolve() == dss_path.resolve()

    rows, cols = (int(value) for value in frames[0]["shape"])
    left, bottom = (float(value) for value in frames[0]["metadata"]["origin"])
    cell = float(frames[0]["cell_size"])
    assert int(received["Raster Rows"]) == rows and int(received["Raster Cols"]) == cols
    assert float(received["Raster Cellsize"]) == cell
    assert float(received["Raster Left"]) == left and float(received["Raster Top"]) == bottom + rows * cell
    assert CRS.from_user_input(received["Projection"]).equals(CRS.from_user_input(frames[0]["crs"]))

    orientations = {
        "south_first_c": lambda grid: grid.reshape(-1, order="C"),
        "north_first_c": lambda grid: np.flipud(grid).reshape(-1, order="C"),
        "south_first_f": lambda grid: grid.reshape(-1, order="F"),
        "north_first_f": lambda grid: np.flipud(grid).reshape(-1, order="F"),
    }
    matches = []
    for name, transform in orientations.items():
        expected = np.stack([transform(grid) for grid in dss_values]).astype(np.float64)
        finite = np.isfinite(expected)
        nodata_ok = (
            np.allclose(hdf_values[~finite], 0.0, rtol=0, atol=5e-6)
            if MATERIALIZED_HDF_NODATA_POLICY == "zero"
            else np.isnan(hdf_values[~finite]).all()
        )
        values_ok = np.isfinite(hdf_values[finite]).all() and np.allclose(
            hdf_values[finite], expected[finite], rtol=0, atol=5e-6,
        )
        if nodata_ok and values_ok:
            matches.append((name, expected, finite))
    assert len(matches) == 1, "Materialized grid orientation is absent or ambiguous"
    orientation, expected_values, finite_values = matches[0]
    finite_cells = finite_values.all(axis=0)
    assert finite_cells.any(), "No finite source-grid cells remain for the cumulative comparison"
    interval_error = np.abs(hdf_values[:, finite_cells] - expected_values[:, finite_cells])
    cumulative_hdf = np.cumsum(hdf_values[:, finite_cells], axis=0)
    cumulative_expected = np.cumsum(expected_values[:, finite_cells], axis=0)
    cumulative_error = np.abs(cumulative_hdf - cumulative_expected)
    np.testing.assert_allclose(cumulative_hdf, cumulative_expected, rtol=0, atol=5e-6)
    tail = hdf_times > storm_end
    if PAD_DRY_TAIL:
        assert tail.any() and np.allclose(hdf_values[tail][:, finite_cells], 0.0, rtol=0, atol=5e-6)
    else:
        assert not tail.any()

    precompute_receipt = {
        "prepared": asdict(prepared),
        "tmp_hdf": str(prepared.tmp_hdf_path),
        "values_shape": list(hdf_values.shape),
        "timestamp_count": int(len(hdf_times)),
        "dss_pathname": dss_paths[0],
        "orientation": orientation,
        "nodata_policy": MATERIALIZED_HDF_NODATA_POLICY,
        "interval_max_abs_error": float(np.max(interval_error)),
        "cumulative_max_abs_error": float(np.max(cumulative_error)),
        "dry_tail_rows": int(tail.sum()),
    }
    (RUN_ROOT / "precompute-receipt.json").write_text(
        json.dumps(precompute_receipt, indent=2, default=str), encoding="utf-8",
    )
    print("Native precompute and materialized forcing passed:", prepared.tmp_hdf_path)
else:
    print("Native precompute not run. Enable only after attachment and DSS read-back review.")


Native precompute not run. Enable only after attachment and DSS read-back review.
